# Modulo 7 · Lezione 6 — Overfitting e Bias-Variance Tradeoff

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Underfitting vs Overfitting |
| 2 | Bias-Variance Tradeoff |
| 3 | Curve di apprendimento |
| 4 | Curve di validazione |
| 5 | Strategie per correggere ciascun problema |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve, validation_curve
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, accuracy_score

np.random.seed(42)
print("Librerie pronte")

---
## 1. Underfitting vs Overfitting

```
UNDERFITTING (bias alto):
  Il modello è troppo semplice per catturare i pattern del dataset.
  → Errore alto sia in train CHE in test.
  → Il modello "non impara".
  Esempio: regressione lineare su dati quadratici.

BUON ADATTAMENTO:
  Il modello cattura i pattern del dataset ma non il rumore.
  → Errore basso in train, errore basso e simile in test.

OVERFITTING (varianza alta):
  Il modello memorizza il rumore del set di training.
  → Errore molto basso in train, errore ALTO in test.
  → Il modello "memorizza invece di imparare".
  Esempio: albero senza limite di profondità.

Il segnale di allarme:
  gap grande tra metrica_train e metrica_test → OVERFITTING
  entrambe le metriche scarse → UNDERFITTING
```

In [ ]:
# Visualizzare i 3 casi
np.random.seed(42)
x = np.linspace(0, 10, 60)
y_true = np.sin(x) * 2
y_noisy = y_true + np.random.normal(0, 0.5, len(x))

X_t = x.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X_t, y_noisy, test_size=0.3, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_rango = np.linspace(0, 10, 300).reshape(-1, 1)

configuraciones = [
    ('Underfitting (grado=1)', 1, 'Underfitting'),
    ('Buon adattamento (grado=4)', 4, 'Buon adattamento'),
    ('Overfitting (grado=15)', 15, 'Overfitting'),
]

for ax, (titulo, grado, tipo) in zip(axes, configuraciones):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(grado, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    pipe.fit(X_tr, y_tr)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, pipe.predict(X_tr)))
    rmse_te = np.sqrt(mean_squared_error(y_te, pipe.predict(X_te)))

    ax.scatter(X_tr, y_tr, alpha=0.5, color='steelblue', s=20, label='Train')
    ax.scatter(X_te, y_te, alpha=0.7, color='orange', s=25, marker='x', label='Test')
    ax.plot(x_rango, pipe.predict(x_rango), 'r-', lw=2)
    ax.plot(x_rango, np.sin(x_rango.ravel())*2, 'g--', lw=1, alpha=0.7, label='Funzione reale')
    ax.set_title(f'{titulo}\nRMSE train={rmse_tr:.3f}, test={rmse_te:.3f}')
    ax.legend(fontsize=8)
    ax.set_ylim(-4, 4)

plt.suptitle('Underfitting, Buon Adattamento e Overfitting', fontsize=12)
plt.tight_layout(); plt.show()

---
## 2. Bias-Variance Tradeoff

```
Errore totale = Bias² + Varianza + Rumore irriducibile

BIAS:
  Errore sistematico dovuto a assunzioni errate del modello.
  Modello semplice → bias alto → underfitting

VARIANZA:
  Sensibilità del modello a piccole variazioni nei dati.
  Modello complesso → varianza alta → overfitting

TRADEOFF: non si possono ridurre entrambi contemporaneamente con lo stesso modello.

          Bias²    Varianza    Errore
Semplice: ALTO     basso       alto (underfitting)
Ottimo:   medio    medio       minimo
Complesso:basso    ALTA        alto (overfitting)

SOLUZIONE: ensemble methods (Random Forest, Gradient Boosting)
  → Riducono la varianza mantenendo il bias basso.
```

In [ ]:
# Dimostrare bias vs varianza con modelli diversi
np.random.seed(42)

def generar_datos(n=50, seed=None):
    rng = np.random.default_rng(seed)
    x   = rng.uniform(0, 10, n)
    y   = np.sin(x) * 2 + rng.normal(0, 0.5, n)
    return x.reshape(-1,1), y

# Simulare 50 esperimenti con campioni diversi
n_experimentos = 50
x_test = np.linspace(0, 10, 100).reshape(-1, 1)
y_real = np.sin(x_test.ravel()) * 2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colores = ['steelblue', 'green', 'coral']
titulos_mod = ['Grado 1 (bias alto)', 'Grado 4 (equilibrato)', 'Grado 12 (varianza alta)']

for ax, grado, titulo, color in zip(axes, [1, 4, 12], titulos_mod, colores):
    predicciones = []
    for i in range(n_experimentos):
        X_i, y_i = generar_datos(n=50, seed=i)
        pipe = Pipeline([
            ('poly', PolynomialFeatures(grado, include_bias=False)),
            ('reg',  Ridge(alpha=0.01))
        ])
        pipe.fit(X_i, y_i)
        predicciones.append(pipe.predict(x_test))

    preds = np.array(predicciones)
    pred_media = preds.mean(axis=0)
    bias2   = np.mean((pred_media - y_real) ** 2)
    varianza = np.mean(preds.var(axis=0))

    for pred in predicciones[:20]:
        ax.plot(x_test, pred, color=color, alpha=0.1, lw=0.8)
    ax.plot(x_test, pred_media, color='black', lw=2, label='Media predizioni')
    ax.plot(x_test, y_real, 'r--', lw=2, label='Funzione reale')
    ax.set_title(f'{titulo}\nBias²={bias2:.3f}, Varianza={varianza:.3f}')
    ax.legend(fontsize=7); ax.set_ylim(-5, 5)

plt.suptitle('Bias-Variance: 50 esperimenti per modello', fontsize=11)
plt.tight_layout(); plt.show()

---
## 3. Curve di apprendimento

```
La curva di apprendimento mostra l'errore (o score) in funzione della
dimensione del set di training.

Come leggerla:

UNDERFITTING:                OVERFITTING:               BUON ADATTAMENTO:
  Train ─── alto               Train ─── basso            Train ─── basso
  Test  ─── alto               Test  ─── alto             Test  ─── basso ≈ Train
  gap piccolo                  gap grande                 gap piccolo

Come correggere?
  Underfitting → modello più complesso, più feature, meno regolarizzazione
  Overfitting  → più dati, meno feature, più regolarizzazione, ensemble
```

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
modelos_lc = [
    ('Underfitting\n(albero depth=1)', DecisionTreeClassifier(max_depth=1, random_state=42)),
    ('Overfitting\n(albero senza limite)', DecisionTreeClassifier(max_depth=None, random_state=42)),
    ('Equilibrato\n(Random Forest)', RandomForestClassifier(50, max_depth=5, random_state=42, n_jobs=-1)),
]

for ax, (titulo, m) in zip(axes, modelos_lc):
    train_sizes, train_scores, test_scores = learning_curve(
        m, X, y,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=5, scoring='accuracy', n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    test_mean  = test_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    test_std   = test_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, 'b-o', ms=5, lw=2, label='Train')
    ax.plot(train_sizes, test_mean,  'r-o', ms=5, lw=2, label='Validazione CV')
    ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color='blue')
    ax.fill_between(train_sizes, test_mean-test_std,  test_mean+test_std,  alpha=0.15, color='red')
    ax.set_ylim(0.5, 1.05)
    ax.set_xlabel('N. di campioni di training')
    ax.set_ylabel('Accuracy')
    ax.set_title(titulo)
    ax.legend(); ax.grid(True, alpha=0.3)
    # Diagnostica automatica
    gap_final = train_mean[-1] - test_mean[-1]
    nivel_train = train_mean[-1]
    if nivel_train < 0.75:
        diag = 'UNDERFITTING'
    elif gap_final > 0.1:
        diag = 'OVERFITTING'
    else:
        diag = 'BUON ADATTAMENTO'
    ax.text(0.05, 0.05, diag, transform=ax.transAxes, fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Curve di apprendimento per la diagnosi di overfitting', fontsize=11)
plt.tight_layout(); plt.show()

---
## 4. Curve di validazione

In [ ]:
# Curva di validazione: score vs iperparametro
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]

train_scores, test_scores = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X, y,
    param_name='max_depth',
    param_range=profundidades,
    cv=5, scoring='accuracy', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
test_mean  = test_scores.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(profundidades, train_mean, 'b-o', ms=5, lw=2, label='Train')
axes[0].plot(profundidades, test_mean,  'r-o', ms=5, lw=2, label='Validazione CV')
mejor_depth = profundidades[np.argmax(test_mean)]
axes[0].axvline(mejor_depth, color='green', ls='--', lw=2,
               label=f'Depth migliore={mejor_depth}')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Curva di validazione — max_depth')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Gap train-test
gap = train_mean - test_mean
axes[1].plot(profundidades, gap, 'purple', lw=2, marker='o', ms=5)
axes[1].axhline(0.1, color='red', ls='--', lw=1, label='Soglia overfitting (0.1)')
axes[1].axvline(mejor_depth, color='green', ls='--', lw=1.5)
axes[1].fill_between(profundidades, 0, gap, alpha=0.2, color='purple')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Gap (train - test)')
axes[1].set_title('Gap di generalizzazione')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Profondità ottimale (score CV più alto): {mejor_depth}")
print(f"  Train: {train_mean[profundidades.index(mejor_depth)]:.4f}")
print(f"  Test CV: {test_mean[profundidades.index(mejor_depth)]:.4f}")

---
## 5. Strategie di correzione

In [ ]:
print("=" * 60)
print("  DIAGNOSI E SOLUZIONI")
print("=" * 60)

print("""
UNDERFITTING (bias alto):
  Sintomo: errore alto sia in train CHE in test, gap piccolo
  Soluzioni:
    1. Modello più complesso (più livelli, più alberi, depth maggiore)
    2. Più feature (feature engineering, feature polinomiali)
    3. Meno regolarizzazione (alpha minore in Ridge/Lasso)
    4. Più tempo di training (più iterazioni)

OVERFITTING (varianza alta):
  Sintomo: errore basso in train, ALTO in test (gap grande)
  Soluzioni:
    1. Più dati (la soluzione più efficace)
    2. Regolarizzazione (Ridge, Lasso, Dropout nelle reti)
    3. Ridurre la complessità (meno depth, meno feature)
    4. Ensemble methods (Random Forest, Gradient Boosting)
    5. Early stopping (fermare il training prima)
    6. Data augmentation (creare dati sintetici)
""")

# Dimostrazione pratica: correggere l'overfitting
np.random.seed(42)
X_small, y_small = make_classification(n_samples=100, n_features=20,
                                        n_informative=5, random_state=42)
X_large, y_large = make_classification(n_samples=2000, n_features=20,
                                        n_informative=5, random_state=42)

arbol = DecisionTreeClassifier(max_depth=None, random_state=42)

print("Dimostrazione: correggere l'overfitting")
print(f"{'Strategia':35} {'Acc_train':>10} {'Acc_test':>10}")

# 1. Baseline (overfitting)
X_tr, X_te, y_tr, y_te = train_test_split(X_small, y_small, test_size=0.3, random_state=42)
arbol.fit(X_tr, y_tr)
print(f"{'Albero senza limite (pochi dati)':35} "
      f"{accuracy_score(y_tr,arbol.predict(X_tr)):>10.4f} "
      f"{accuracy_score(y_te,arbol.predict(X_te)):>10.4f}")

# 2. Limitare la profondità
arbol_lim = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
print(f"{'Albero depth=4':35} "
      f"{accuracy_score(y_tr,arbol_lim.predict(X_tr)):>10.4f} "
      f"{accuracy_score(y_te,arbol_lim.predict(X_te)):>10.4f}")

# 3. Più dati
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_large, y_large, test_size=0.3, random_state=42)
arbol.fit(X_tr2, y_tr2)
print(f"{'Albero senza limite (molti dati)':35} "
      f"{accuracy_score(y_tr2,arbol.predict(X_tr2)):>10.4f} "
      f"{accuracy_score(y_te2,arbol.predict(X_te2)):>10.4f}")

# 4. Random Forest
rf = RandomForestClassifier(100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print(f"{'Random Forest (pochi dati)':35} "
      f"{accuracy_score(y_tr,rf.predict(X_tr)):>10.4f} "
      f"{accuracy_score(y_te,rf.predict(X_te)):>10.4f}")

---
## Pratica

### Esercizio 1 — Diagnosticare un modello

**Consegna:** Dato questo modello e dataset:

```python
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
```

1. Traccia la curva di apprendimento per: DecisionTree depth=1, depth=5, depth=None, RandomForest 100
2. Diagnostica ciascun modello (underfitting / overfitting / buon adattamento)
3. Quale consiglieresti per la produzione?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer()
X, y = data.data, data.target

modelos = [
    ('DT depth=1',      DecisionTreeClassifier(max_depth=1, random_state=42)),
    ('DT depth=5',      DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('DT senza limite', DecisionTreeClassifier(random_state=42)),
    ('RF 100',          RandomForestClassifier(100, random_state=42, n_jobs=-1)),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (titulo, m) in zip(axes, modelos):
    sizes, tr_sc, te_sc = learning_curve(
        m, X, y, train_sizes=np.linspace(0.1,1,8), cv=5, scoring='accuracy', n_jobs=-1)
    tr_m, te_m = tr_sc.mean(axis=1), te_sc.mean(axis=1)
    ax.plot(sizes, tr_m, 'b-o', ms=4, label='Train')
    ax.plot(sizes, te_m, 'r-o', ms=4, label='Val CV')
    ax.set_title(titulo); ax.set_ylim(0.7, 1.02)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    # Diagnostica
    gap = tr_m[-1] - te_m[-1]
    if tr_m[-1] < 0.85: diag = 'UNDERFITTING'
    elif gap > 0.08:    diag = 'OVERFITTING'
    else:               diag = 'BUON ADATTAMENTO'
    ax.text(0.05, 0.05, diag, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Diagnosi: Breast Cancer Dataset', fontsize=11)
plt.tight_layout(); plt.show()
print("Raccomandazione: Random Forest — buon score, senza overfitting significativo.")
```
</details>

---
### Esercizio 2 — Curva di validazione per Ridge

**Consegna:** Genera un dataset di regressione con 50 feature (30 irrilevanti):

```python
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=200, n_features=50, n_informative=20,
                       noise=20, random_state=42)
```

1. Traccia la curva di validazione per Ridge con alpha: `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`
2. Identifica l'alpha ottimale
3. Confronta Ridge ottimale vs LinearRegression: R² sul test
4. Mostra il gap train-test per ogni alpha

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score

X, y = make_regression(n_samples=200, n_features=50, n_informative=20, noise=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
tr_sc, te_sc = validation_curve(
    Ridge(), X_tr, y_tr,
    param_name='alpha', param_range=alphas,
    cv=5, scoring='r2', n_jobs=-1
)

tr_m, te_m = tr_sc.mean(axis=1), te_sc.mean(axis=1)
mejor_alpha = alphas[np.argmax(te_m)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].semilogx(alphas, tr_m, 'b-o', ms=5, label='Train')
axes[0].semilogx(alphas, te_m, 'r-o', ms=5, label='Val CV')
axes[0].axvline(mejor_alpha, color='green', ls='--', label=f'α ottimale={mejor_alpha}')
axes[0].set_xlabel('Alpha'); axes[0].set_ylabel('R²')
axes[0].set_title('Curva di validazione Ridge'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

gap = tr_m - te_m
axes[1].semilogx(alphas, gap, 'purple', lw=2, marker='o', ms=5)
axes[1].axvline(mejor_alpha, color='green', ls='--')
axes[1].set_xlabel('Alpha'); axes[1].set_ylabel('Gap train-test')
axes[1].set_title('Gap di generalizzazione'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Confronto sul test
for nombre, m in [('LinearRegression', LinearRegression()),
                  (f'Ridge α={mejor_alpha}', Ridge(alpha=mejor_alpha))]:
    m.fit(X_tr, y_tr)
    r2 = r2_score(y_te, m.predict(X_te))
    r2_tr = r2_score(y_tr, m.predict(X_tr))
    print(f"{nombre:25}: R²_train={r2_tr:.4f}, R²_test={r2:.4f}")
print(f"\nRidge ottimale riduce il gap grazie alla regolarizzazione.")
```
</details>

---
### Esercizio 3 — Più dati o modello più semplice?

**Consegna:** Un'azienda ha 200 campioni e il suo albero (depth=10) presenta overfitting. Prima di raccogliere più dati (costoso), vuole sapere se la soluzione più semplice (limitare l'albero) funziona.

```python
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, n_features=20, n_informative=8, random_state=42)
X_200, _, y_200, _ = train_test_split(X, y, train_size=200, random_state=42)
X_te, y_te = X[1800:], y[1800:]  # test fisso
```

1. Traccia le curve di apprendimento per albero depth=10 e RF 100
2. A partire da quanti campioni l'albero depth=10 smette di avere overfitting?
3. Cosa consigli: più dati o Random Forest con 200 campioni?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=2000, n_features=20, n_informative=8, random_state=42)

# Curve di apprendimento con tutti i dati disponibili
modelos = [
    ('Albero depth=10', DecisionTreeClassifier(max_depth=10, random_state=42)),
    ('RF 100', RandomForestClassifier(100, random_state=42, n_jobs=-1)),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (titulo, m) in zip(axes, modelos):
    sizes, tr_sc, te_sc = learning_curve(
        m, X, y, train_sizes=np.linspace(0.05, 0.9, 10),
        cv=5, scoring='accuracy', n_jobs=-1)
    tr_m, te_m = tr_sc.mean(axis=1), te_sc.mean(axis=1)
    ax.plot(sizes, tr_m, 'b-o', ms=4, label='Train')
    ax.plot(sizes, te_m, 'r-o', ms=4, label='Val CV')
    # Segnare dove il gap scende sotto 0.05
    gaps = tr_m - te_m
    bajo_umbral = np.where(gaps < 0.05)[0]
    if len(bajo_umbral) > 0:
        n_umbral = sizes[bajo_umbral[0]]
        ax.axvline(n_umbral, color='green', ls='--', label=f'Gap<5% da n={n_umbral:.0f}')
    ax.set_title(titulo); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlabel('N. campioni'); ax.set_ylabel('Accuracy')

plt.tight_layout(); plt.show()

# Confronto con 200 campioni
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
X_200 = X_tr[:200]; y_200 = y_tr[:200]

for nombre, m in modelos:
    m.fit(X_200, y_200)
    acc = accuracy_score(y_te, m.predict(X_te))
    print(f"{nombre}: accuracy sul test con 200 campioni = {acc:.4f}")

print("\nRaccomandazione: RF con 200 campioni è di solito migliore dell'albero.")
print("Il bagging compensa la scarsità di dati.")
```
</details>

---
### Esercizio 4 — Ricetta di diagnosi

**Consegna:** Dato questo modello in produzione che segnala problemi:

```python
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=150, n_features=30, n_informative=5, noise=30, random_state=7)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
from sklearn.linear_model import LinearRegression
m = LinearRegression().fit(X_tr, y_tr)
```

1. Calcola R² su train e test → qual è la diagnosi?
2. Traccia la curva di apprendimento
3. Proponi e prova 2 soluzioni distinte
4. Quale migliora di più l'R² sul test?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X, y = make_regression(n_samples=150, n_features=30, n_informative=5, noise=30, random_state=7)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

m = LinearRegression().fit(X_tr, y_tr)

# 1. Diagnosi
r2_tr = r2_score(y_tr, m.predict(X_tr))
r2_te = r2_score(y_te, m.predict(X_te))
print(f"R² train: {r2_tr:.4f}")
print(f"R² test:  {r2_te:.4f}")
print(f"Gap:      {r2_tr - r2_te:.4f}")
print("→ OVERFITTING: R² molto basso nel test rispetto al train")
print("  Causa: n=150, p=30 → molte feature rispetto ai campioni")

# 2. Curva di apprendimento
sizes, tr_sc, te_sc = learning_curve(m, X, y, train_sizes=np.linspace(0.1,0.9,8),
                                      cv=5, scoring='r2', n_jobs=-1)
plt.figure(figsize=(8,4))
plt.plot(sizes, tr_sc.mean(1), 'b-o', label='Train')
plt.plot(sizes, te_sc.mean(1), 'r-o', label='Val CV')
plt.xlabel('Campioni'); plt.ylabel('R²'); plt.title('Curva di apprendimento')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# 3. Soluzioni
for nombre, mod in [
    ('LinearRegression', LinearRegression()),
    ('Ridge α=10',       Ridge(alpha=10)),
    ('Lasso α=1',        Lasso(alpha=1)),
    ('RandomForest',     RandomForestRegressor(100, random_state=42, n_jobs=-1)),
]:
    mod.fit(X_tr, y_tr)
    r2 = r2_score(y_te, mod.predict(X_te))
    r2_tr_i = r2_score(y_tr, mod.predict(X_tr))
    print(f"{nombre:20}: R²_train={r2_tr_i:.4f}, R²_test={r2:.4f}")
```
</details>

---
### Esercizio 5 (sfida) — Funzione di diagnosi automatica

**Consegna:** Implementa `diagnosticar_modelo(modelo, X, y, cv=5)` che:

1. Calcola e traccia la curva di apprendimento
2. Diagnostica automaticamente: underfitting / overfitting / buon adattamento
3. Propone le due soluzioni più adatte in base alla diagnosi
4. Restituisce un dict con `{'diagnostico': ..., 'score_train': ..., 'score_val': ..., 'gap': ..., 'soluciones': [...]}`

Provala con 3 modelli distinti.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

SOLUCIONES = {
    'underfitting': [
        'Modello più complesso (più profondità, più alberi)',
        'Più feature o feature polinomiali',
        'Meno regolarizzazione (alpha minore)',
    ],
    'overfitting': [
        'Raccogliere più dati di training',
        'Aumentare la regolarizzazione (alpha maggiore)',
        'Ridurre la complessità del modello',
        'Usare ensemble (Random Forest)',
    ],
    'buen ajuste': [
        'Il modello generalizza bene',
        'Considera di aggiustare la soglia di classificazione se rilevante',
    ],
}

def diagnosticar_modelo(modelo, X, y, cv=5, scoring='accuracy', ax=None):
    sizes, tr_sc, te_sc = learning_curve(
        modelo, X, y,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=cv, scoring=scoring, n_jobs=-1
    )
    tr_m, te_m = tr_sc.mean(axis=1), te_sc.mean(axis=1)
    score_train = tr_m[-1]
    score_val   = te_m[-1]
    gap         = score_train - score_val

    # Diagnosi
    umbral_malo = 0.75 if scoring == 'accuracy' else 0.5
    if score_train < umbral_malo:   diag = 'underfitting'
    elif gap > 0.08:                diag = 'overfitting'
    else:                           diag = 'buen ajuste'

    # Plot
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sizes, tr_m, 'b-o', ms=5, lw=2, label='Train')
    ax.plot(sizes, te_m, 'r-o', ms=5, lw=2, label='Val CV')
    ax.fill_between(sizes, tr_sc.min(1), tr_sc.max(1), alpha=0.1, color='blue')
    ax.fill_between(sizes, te_sc.min(1), te_sc.max(1), alpha=0.1, color='red')
    ax.set_title(f'{type(modelo).__name__}\n{diag.upper()}')
    ax.set_xlabel('Campioni'); ax.set_ylabel(scoring)
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.05, diag.upper(), transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat'))

    return {
        'diagnostico': diag,
        'score_train': round(score_train, 4),
        'score_val':   round(score_val, 4),
        'gap':         round(gap, 4),
        'soluciones':  SOLUCIONES[diag][:2],
    }

# Prova
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=500, n_features=20, n_informative=8, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(axes, [
    DecisionTreeClassifier(max_depth=1, random_state=42),
    DecisionTreeClassifier(max_depth=None, random_state=42),
    RandomForestClassifier(100, random_state=42, n_jobs=-1),
]):
    resultado = diagnosticar_modelo(m, X, y, ax=ax)
    print(f"\n{type(m).__name__}:")
    for k, v in resultado.items():
        print(f"  {k}: {v}")

plt.tight_layout(); plt.show()
```
</details>

---
## Riepilogo

```python
from sklearn.model_selection import learning_curve, validation_curve
import numpy as np

# Curva di apprendimento (diagnosi in base alla dimensione dei dati)
sizes, tr_scores, te_scores = learning_curve(
    modelo, X, y,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5, scoring='accuracy'
)

# Curva di validazione (diagnosi in base all'iperparametro)
tr_scores, te_scores = validation_curve(
    modelo, X, y,
    param_name='max_depth', param_range=[1,2,3,5,10],
    cv=5, scoring='accuracy'
)

# Diagnosi rapida:
# score_train alto, score_val basso   → OVERFITTING
# entrambi bassi                       → UNDERFITTING
# entrambi alti e simili               → BUON ADATTAMENTO
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 7** impareremo la **Cross-Validation** e **GridSearchCV**: gli strumenti standard per valutare i modelli e trovare i migliori iperparametri in modo affidabile.